# NumPy for KMeans: 3 Key Operations

In [1]:
import numpy as np

## 1. How to Measure Distance Between Two Points

Euclidean distance: subtract, square, sum, square root. `np.linalg.norm` does it in one call.

In [2]:
x1 = np.array([1, 2, 3])
x2 = np.array([4, 6, 3])

# Step by step
diff = x1 - x2                        # [-3, -4, 0]
squared = diff ** 2                    # [9, 16, 0]
summed = np.sum(squared)               # 25
distance = np.sqrt(summed)             # 5.0

print("x1:        ", x1)
print("x2:        ", x2)
print("difference:", diff)
print("squared:   ", squared)
print("sum:       ", summed)
print("sqrt:      ", distance)

# Shortcut: np.linalg.norm does all steps at once
print("\nnp.linalg.norm(x1 - x2) =", np.linalg.norm(x1 - x2))

x1:         [1 2 3]
x2:         [4 6 3]
difference: [-3 -4  0]
squared:    [ 9 16  0]
sum:        25
sqrt:       5.0

np.linalg.norm(x1 - x2) = 5.0


## 2. How to Assign Points to the Nearest Centroid

Compute distance from every point to each centroid, then pick the closest one with `np.argmin`.

In [3]:
# 5 data points in 2D
X = np.array([[1, 2], [2, 1], [6, 7], [7, 8], [8, 6]])

# 2 initial centroids -- deliberately placed poorly (both near the left)
c0 = np.array([2.0, 2.0])
c1 = np.array([3.0, 3.0])

# Distance from every point to each centroid (broadcasting + norm)
dist_to_c0 = np.linalg.norm(X - c0, axis=1)  # shape (5,)
dist_to_c1 = np.linalg.norm(X - c1, axis=1)  # shape (5,)

# Build distance table: N x K
dist_table = np.column_stack([dist_to_c0, dist_to_c1])
print("Distance table (rows=points, cols=centroids):")
print(dist_table.round(2))

# Assign each point to nearest centroid
labels = np.argmin(dist_table, axis=1)
print("\nAssignments (0=near c0, 1=near c1):", labels)

Distance table (rows=points, cols=centroids):
[[1.   2.24]
 [1.   2.24]
 [6.4  5.  ]
 [7.81 6.4 ]
 [7.21 5.83]]

Assignments (0=near c0, 1=near c1): [0 0 1 1 1]


## 3. How Centroids Update

After assigning points, each centroid **moves** to the mean of its cluster members. This is the core of KMeans -- centroids keep shifting until they stop moving.

In [4]:
# Using X, labels, c0, c1 from above
old_centroids = np.array([c0, c1])

# Get members of each cluster
cluster_0 = X[labels == 0]
cluster_1 = X[labels == 1]
print("Cluster 0 members:", cluster_0.tolist())
print("Cluster 1 members:", cluster_1.tolist())

# New centroid = mean of members
new_c0 = cluster_0.mean(axis=0)
new_c1 = cluster_1.mean(axis=0)

print("\nCentroid update:")
print(f"  c0: {c0} --> {new_c0}  (moved {np.linalg.norm(new_c0 - c0):.2f})")
print(f"  c1: {c1} --> {new_c1}  (moved {np.linalg.norm(new_c1 - c1):.2f})")
print(f"\n  c1 jumped from {c1} to {new_c1} -- that's a big move!")

# Check convergence
print("\nConverged?", np.allclose(old_centroids, np.array([new_c0, new_c1])))
print("  --> Not yet. Repeat until centroids stop moving.")

Cluster 0 members: [[1, 2], [2, 1]]
Cluster 1 members: [[6, 7], [7, 8], [8, 6]]

Centroid update:
  c0: [2. 2.] --> [1.5 1.5]  (moved 0.71)
  c1: [3. 3.] --> [7. 7.]  (moved 5.66)

  c1 jumped from [3. 3.] to [7. 7.] -- that's a big move!

Converged? False
  --> Not yet. Repeat until centroids stop moving.


## Full Picture: Two KMeans Iterations

Distance --> assign --> update centroids --> check if done. Watch how centroids converge in 2 rounds.

In [5]:
# Data and bad initial centroids
X = np.array([[1, 2], [2, 1], [6, 7], [7, 8], [8, 6]])
centroids = np.array([[2.0, 2.0], [3.0, 3.0]])  # both near the left!
K = 2

print("=== Iteration 1 ===")
# Step 1: distances
dists = np.zeros((len(X), K))
for k in range(K):
    dists[:, k] = np.linalg.norm(X - centroids[k], axis=1)

# Step 2: assign
labels = np.argmin(dists, axis=1)
print("Labels:", labels)

# Step 3: update centroids
new_centroids = np.zeros_like(centroids)
for k in range(K):
    new_centroids[k] = X[labels == k].mean(axis=0)

for k in range(K):
    moved = np.linalg.norm(new_centroids[k] - centroids[k])
    print(f"  c{k}: {centroids[k]} --> {new_centroids[k]}  (moved {moved:.2f})")

# Step 4: converged?
print("Converged:", np.allclose(centroids, new_centroids))

# --- Do it again with updated centroids ---
centroids = new_centroids.copy()
print("\n=== Iteration 2 ===")
for k in range(K):
    dists[:, k] = np.linalg.norm(X - centroids[k], axis=1)
labels = np.argmin(dists, axis=1)
print("Labels:", labels)

new_centroids2 = np.zeros_like(centroids)
for k in range(K):
    new_centroids2[k] = X[labels == k].mean(axis=0)

for k in range(K):
    moved = np.linalg.norm(new_centroids2[k] - centroids[k])
    print(f"  c{k}: {centroids[k]} --> {new_centroids2[k]}  (moved {moved:.2f})")

print("Converged:", np.allclose(centroids, new_centroids2), " --> Done!")

=== Iteration 1 ===
Labels: [0 0 1 1 1]
  c0: [2. 2.] --> [1.5 1.5]  (moved 0.71)
  c1: [3. 3.] --> [7. 7.]  (moved 5.66)
Converged: False

=== Iteration 2 ===
Labels: [0 0 1 1 1]
  c0: [1.5 1.5] --> [1.5 1.5]  (moved 0.00)
  c1: [7. 7.] --> [7. 7.]  (moved 0.00)
Converged: True  --> Done!
